<a href="https://colab.research.google.com/github/Vermont-Complex-Systems/storywrangler/blob/main/notebooks/mcp_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MCP and Agents demo
`uv` is installed out of the box, while  `uvx` let you run a command without installing it. This pretty cool. I means we can run the `list-sections` command from our [storywrangler-mcp](https://pypi.org/project/storywrangler-mcp/) lib without installing anything directly on colab.

Our agentic setup is inspired by the [svelte-mcp](https://svelte.dev/docs/ai/overview).

In [ ]:
# look up the command available from the mcp
!uvx storywrangler-mcp --help

usage: storywrangler-mcp [-h]
                         {validate-submission,list-sections,get-documentation,list-datasets,get-dataset}
                         ...

Storywrangler MCP server (no arguments) or CLI (subcommands).

positional arguments:
  {validate-submission,list-sections,get-documentation,list-datasets,get-dataset}
    validate-submission
                        Dry-run a DatasetCreate payload
    list-sections       List documentation sections
    get-documentation   Fetch documentation section(s)
    list-datasets       List registered datasets
    get-dataset         Get one dataset's registry metadata

options:
  -h, --help            show this help message and exit


In [ ]:
# list-sections return sections from
# https://storywrangler.uvm.edu/ with
# various use cases, here generated by Claude
!uvx storywrangler-mcp list-sections

HTTP Request: GET https://storywrangler.uvm.edu/sections.json "HTTP/1.1 200 OK"
Available documentation sections:
- title: Getting started, use_cases: setup, install SDK, storywrangler new, scaffold project, core concepts, domains, datasets, registry, register first dataset, DatasetCreate, query, allotax, rtd, base URL, path: getting-started
- title: The challenge of building digital commons for academia, use_cases: motivation, why storywrangler, academic data commons, collective attention, background, path: manifesto
- title: Authentication, use_cases: API keys, Bearer token, login, roles, admin bootstrap, path: authentication
- title: Agents & MCP, use_cases: LLM agents, MCP server, model context protocol, llms.txt, machine-readable docs, storywrangler-mcp, uvx, list-datasets, get-dataset, validate-submission, Claude skills, storywrangler-analyst, storywrangler-submission, plugin marketplace, agentic setup, path: llms
- title: Querying Datasets, use_cases: querying data, filters, ent

Once agents know about the usecases, they can dive into any one of the section. Since docs are markdown files, agents like Claude can take advantage of [progressive disclosure](https://platform.claude.com/docs/en/agents-and-tools/agent-skills/best-practices#progressive-disclosure-patterns) to not clutter their context window.

In [ ]:
!uvx storywrangler-mcp get-documentation register-big-data

HTTP Request: GET https://storywrangler.uvm.edu/sections.json "HTTP/1.1 200 OK"
HTTP Request: GET https://storywrangler.uvm.edu/register-big-data/llms.txt "HTTP/1.1 200 OK"
# Registering big data

Flat parquet is right up to a few gigabytes — don't partition small data. Past that, and once queries slice into the whole (one country, one granularity, one date range), switch `data_format` to `parquet_hive`: DuckDB then prunes partitions from directory names instead of opening files.

This page covers the two at-scale declarations: hive-partitioned storage and hash-bucketed partitions. The field-by-field registration basics are in [registering a dataset](/register); the pipeline-side craft — choosing partition keys, sizing files — is in [building a pipeline](/pipelines).

## Hive-partitioned storage

Set `data_format` to `parquet_hive` to enable [hive_partitioning](https://duckdb.org/docs/current/data/partitioning/hive_partitioning). All hive partition levels are auto-discovered from the d

In [ ]:
 !uvx storywrangler-mcp list-datasets

HTTP Request: GET https://api.storywrangler.uvm.edu/registry/ "HTTP/1.1 200 OK"
18 registered dataset(s):
- babynames/ngrams [parquet] — Baby names by popularity, year, and location with entity mappings
- open-academic-analytics/academic-research-groups [parquet] — UVM faculty roster with OpenAlex IDs, departments, and research group metadata. Hand-annotated from annual UVM payroll PDFs.
- open-academic-analytics/authors [parquet] — Per-author summary: current career age, last publication year, and research group status. One row per ego author.
- open-academic-analytics/coauthors [parquet] — UVM faculty coauthor relationships with age categories and collaboration counts. One row per coauthor per publication year per ego author.
- open-academic-analytics/papers [parquet] — UVM faculty papers with UMAP embeddings, citation metrics, topic data, and department/college metadata. One row per paper per ego author.
- open-academic-analytics/training [parquet] — UVM faculty collaboration traini

In [ ]:
 !uvx storywrangler-mcp get-dataset babynames ngrams

HTTP Request: GET https://api.storywrangler.uvm.edu/registry/babynames/ngrams "HTTP/1.1 200 OK"
{
  "catalog": "vcsi",
  "domain": "babynames",
  "dataset_id": "ngrams",
  "version": "latest",
  "schema_version": "0.1.0",
  "data_location": [
    "/users/j/s/jstonge1/babynames/metadata.ducklake.files/main/babynames/ducklake-019d25be-a8d4-7e9c-aa13-3847bd1d8fe4.parquet",
    "/users/j/s/jstonge1/babynames/metadata.ducklake.files/main/babynames/ducklake-019d25be-c181-7b2c-a00b-d32c32d3cf70.parquet"
  ],
  "data_format": "parquet",
  "description": "Baby names by popularity, year, and location with entity mappings",
  "manifest": {
    "availability": {
      "quebec": {
        "min": "1980",
        "max": "2024",
        "types": 1700
      },
      "united_states": {
        "min": "1880",
        "max": "2024",
        "types": 29225
      }
    }
  },
  "data_schema": {
    "types": "VARCHAR",
    "sex": "VARCHAR",
    "counts": "INTEGER",
    "year": "INTEGER",
    "geo": "VARCHAR"

This is basically what you're installing when adding the following to your editor
```json
{
  "mcpServers": {
    "storywrangler": {
      "command": "uvx",
      "args": ["storywrangler-mcp"]
    }
  }
}
```

## Agent Skills

In [ ]:
# skills are simply hosted on github
!wget https://raw.githubusercontent.com/Vermont-Complex-Systems/storywrangler/refs/heads/wiki-hedo/.claude/skills/storywrangler-submitter/SKILL.md

--2026-07-21 15:44:49--  https://raw.githubusercontent.com/Vermont-Complex-Systems/storywrangler/refs/heads/wiki-hedo/.claude/skills/storywrangler-submitter/SKILL.md
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6088 (5.9K) [text/plain]
Saving to: ‘SKILL.md’

SKILL.md            100%[===================>]   5.95K  --.-KB/s    in 0s      

2026-07-21 15:44:49 (64.6 MB/s) - ‘SKILL.md’ saved [6088/6088]



Together with the MCP, we ship a skill. We think as our sklls as distilled docs of our framework, encoding the key principles that are less volatile than the MCP. A key benefit from the SKILLs is how they can trigger in particular context without users necessary asking for it. For a framework like Storywrangler, it is particularly useful as skills can help guide users with their agents to do things like we want to. Below, we guide the agents to keep users in the loop instead of taking the wheels, inviting them to ask their human instead of claudesplaining.

In [ ]:
!pip install python-frontmatter

In [ ]:
import frontmatter

from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.table import Table

console = Console()

post = frontmatter.load("SKILL.md")

# YAML header is now a dict in post.metadata
meta = Table.grid(padding=(0, 1))
meta.add_column(style="bold cyan")
meta.add_column()
for key, value in post.metadata.items():
    meta.add_row(f"{key}:", str(value))
console.print(Panel(meta, title="frontmatter", border_style="dim"))

# Body is plain markdown
console.print(Markdown(post.content))

╭────────────────────────────────────────────────── frontmatter ──────────────────────────────────────────────────╮
│ name:        storywrangler-submitter                                                                            │
│ description: Interactive guide for getting data onto the Storywrangler platform — writing or improving a        │
│              DatasetCreate payload, deciding between serving through an existing endpoint type or disseminating │
│              behind a bespoke endpoint, validating before registration, and verifying what the server derived.  │
│              Use whenever a user wants to submit, register, share, or publish a dataset on Storywrangler, needs │
│              help writing the registration payload, asks whether their data is submittable, or wants to fix or  │
│              optimize an existing submission — even if they never say the word "submission". Not for building   │
│              the extract/transform pipeline that produces the data.                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                      Submitting datasets to Storywrangler                                       ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

The payload is built in conversation with its author: propose, confirm, then build. Start from the smallest valid  
payload and add only what earns its place.                                                                         


                                        The shared core (every submission)                                         

Whatever else happens, every payload needs:                                                                        

 • identity: catalog, domain, dataset_id — check the registry first (GET /registry/domains, list-datasets): an     
   existing entry means update, not create.                                                                        
 • location: data_location + data_format (parquet unless the data is a hive-partitioned tree).                     
 • governance: description, ownership.owner_group, ownership.contact, lineage.repo.                                

An author with a single data.parquet should get from "here's my file" to a valid payload in two or three questions.
Governance is the only part they must be asked; identity can usually be proposed from context.                     


                                The fork: existing endpoint type, or dissemination?                                

One question then decides how much more of the contract matters:                                                   

Existing endpoint type — the data's shape matches a recurring endpoint type (types-counts: rank distributions;     
time-series: tabular measures), so generic endpoints can serve it directly. Declare endpoint_schema plus what the  
type requires:                                                                                                     

 • types-counts — columns default to types/counts; declare type_column/count_column only when the data's names     
   differ (declare around the data, never rename it). Requires one comparison axis: entity_mapping or              
   transform.filter_dimensions.                                                                                    
 • time-series — requires transform.time_dimension and at least one filter_dimension.                              

Dissemination — the platform hosts the data behind an endpoint written for it (e.g. /wikimedia/semantic-timeseries,
essentially SELECT * WHERE country = ?). The endpoint hardcodes the data's shape, so no endpoint_schema is needed  
and the author doesn't have to learn those fields — the core payload is enough to register. Serving needs the      
bespoke endpoint to exist: a PR to the Storywrangler repo, which can come after registration.                      

When the data plainly reduces to (type, count) rows or GROUP-BY-able measures, recommend the endpoint-type path;   
wide bespoke shapes (score columns, JSON maps) point to dissemination. Recommend, don't decide.                    


                                 Performance declarations (both paths, all opt-in)                                 

Everything beyond the core exists for one reason: Storywrangler's query layer is a thin DuckDB layer over parquet, 
and each declaration lets it read less. Offer each as one question with its payoff attached:                       

 • transform.time_dimension → time-range pruning, ?dates= queries, and coverage auto-derived at registration.      
 • transform.filter_dimensions → categorical slicing on in-file columns (omitting the parameter = aggregate over   
   all values).                                                                                                    
 • entity_mapping (+ entities rows when